In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Best model (can be a Pipeline with LogisticRegression inside)
best_model = grid_search.best_estimator_

# Set up StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), start=1):
    # Use .iloc for pandas DataFrame/Series indexing
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Important: clone to get a fresh copy of the best model each time
    model = clone(best_model)
    # Fix: Train on X_train and y_train, not the entire dataset
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_val)

    # Probabilities (for ROC-AUC)
    y_proba = model.predict_proba(X_val)[:, 1]

    fold_results.append({
        "Fold": fold,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred, zero_division=0),
        "Recall": recall_score(y_val, y_pred, zero_division=0),
        "F1": f1_score(y_val, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_val, y_proba)
    })

# Put in a DataFrame
cv_results_df = pd.DataFrame(fold_results)

# Create a scatter plot for each metric across folds
plt.figure(figsize=(12, 8))
sns.set_style("whitegrid")

# Melt the dataframe to get it in the right format for seaborn
melted_df = pd.melt(cv_results_df, id_vars=['Fold'], 
                    value_vars=['Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC'],
                    var_name='Metric', value_name='Score')

# Create the scatter plot
sns.scatterplot(data=melted_df, x='Fold', y='Score', hue='Metric', s=100)

# Add lines connecting points of the same metric
for metric in ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC']:
    metric_data = cv_results_df[['Fold', metric]]
    plt.plot(metric_data['Fold'], metric_data[metric], 'o-', alpha=0.7)

plt.title('Model Performance Metrics Across Folds', fontsize=16)
plt.xlabel('Fold', fontsize=14)
plt.ylabel('Score', fontsize=14)
plt.xticks(cv_results_df['Fold'])
plt.ylim(0, 1.05)
plt.legend(title='Metric', title_fontsize=12, fontsize=10)
plt.tight_layout()
plt.show()

# Display the results dataframe
cv_results_df